# Stage 13: Productization - Submission

This self-contained exercise trains a two-feature linear regression model, saves it with `joblib`, serves it through Flask, and verifies both API routes plus an invalid request. The model uses synthetic data only and is intended to demonstrate deployment mechanics, not to support a real-world decision.

## 1. Train, save, and reload the model

The random seed and generation parameters match the assignment. The output below proves that the serialized model can be loaded and used independently of the fitted in-memory object.

In [1]:
from pathlib import Path
import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

X, y = make_regression(
    n_samples=100, n_features=2, noise=0.1, random_state=42
)
model = LinearRegression().fit(X, y)

Path('model').mkdir(parents=True, exist_ok=True)
joblib.dump(model, 'model/model.pkl')

reloaded_model = joblib.load('model/model.pkl')
example_features = [[0.1, 0.2]]
print('Saved model/model.pkl:', Path('model/model.pkl').exists())
print('Reloaded-model prediction:', float(reloaded_model.predict(example_features)[0]))

Saved model/model.pkl: True
Reloaded-model prediction: 23.58961171297328


## 2. Write the Flask application

The model is loaded once at module import, before either route is called. Both routes reuse the same object and share validation logic so malformed input returns JSON with status 400.

In [2]:
%%writefile app.py
"""Flask API for the Stage 13 two-feature regression model."""

from pathlib import Path
import math

from flask import Flask, jsonify, request
import joblib


# Load the model once when the application starts, never inside a route.
MODEL_PATH = Path(__file__).resolve().parent / "model" / "model.pkl"
model = joblib.load(MODEL_PATH)
app = Flask(__name__)


def _validate_features(values):
    """Return two finite floats or an error message for invalid input."""
    if not isinstance(values, (list, tuple)) or len(values) != 2:
        return None, "'features' must be a list containing exactly two values."

    try:
        features = [float(value) for value in values]
    except (TypeError, ValueError):
        return None, "Both feature values must be numeric."

    if not all(math.isfinite(value) for value in features):
        return None, "Both feature values must be finite numbers."
    return features, None


def _prediction_response(features):
    """Predict one row and return a JSON response."""
    prediction = float(model.predict([features])[0])
    return jsonify({"prediction": prediction})


@app.route("/predict", methods=["POST"])
def predict_post():
    """Predict from a JSON body such as {"features": [0.1, 0.2]}."""
    data = request.get_json(silent=True)
    if not isinstance(data, dict) or "features" not in data:
        return jsonify({"error": "JSON body must include a 'features' key."}), 400

    features, error = _validate_features(data["features"])
    if error:
        return jsonify({"error": error}), 400
    return _prediction_response(features)


@app.route("/predict/<f1>/<f2>", methods=["GET"])
def predict_get(f1, f2):
    """Predict from two numeric URL path parameters."""
    features, error = _validate_features([f1, f2])
    if error:
        return jsonify({"error": error}), 400
    return _prediction_response(features)


if __name__ == "__main__":
    # Port 5050 avoids the macOS service that can occupy port 5000.
    app.run(host="127.0.0.1", port=5050, debug=False)


Overwriting app.py


## 3. Launch the local API

The notebook starts the API as a separate process and waits until port 5050 accepts requests. This makes the following live tests reproducible when the notebook is run top to bottom.

In [3]:
import subprocess
import sys
import time
import requests

server_process = subprocess.Popen(
    [sys.executable, 'app.py'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
BASE_URL = 'http://127.0.0.1:5050'

for _ in range(30):
    try:
        probe = requests.get(BASE_URL + '/predict/0/0', timeout=0.5)
        if probe.status_code == 200:
            break
    except requests.RequestException:
        time.sleep(0.2)
else:
    server_process.terminate()
    raise RuntimeError('Flask server did not start within the expected time.')

print('Flask API is ready at', BASE_URL)

Flask API is ready at http://127.0.0.1:5050


## 4. Call both routes and test bad input

The first two responses should agree because they send the same features through different interfaces. The third request deliberately supplies a non-numeric path value and must return HTTP 400 with a JSON error rather than a traceback.

In [4]:
post_response = requests.post(
    BASE_URL + '/predict', json={'features': [0.1, 0.2]}, timeout=5
)
get_response = requests.get(BASE_URL + '/predict/0.1/0.2', timeout=5)
bad_response = requests.get(BASE_URL + '/predict/abc/0.2', timeout=5)

print('POST /predict         ', post_response.status_code, post_response.json())
print('GET /predict/0.1/0.2 ', get_response.status_code, get_response.json())
print('GET /predict/abc/0.2 ', bad_response.status_code, bad_response.json())

assert post_response.status_code == 200
assert get_response.status_code == 200
assert bad_response.status_code == 400
assert post_response.json() == get_response.json()
assert 'error' in bad_response.json()

POST /predict          200 {'prediction': 23.58961171297328}
GET /predict/0.1/0.2  200 {'prediction': 23.58961171297328}
GET /predict/abc/0.2  400 {'error': 'Both feature values must be numeric.'}


## 5. Additional POST validation

The assignment specifically requires missing or incorrectly sized `features` values to fail cleanly. These assertions cover both cases.

In [5]:
missing_response = requests.post(BASE_URL + '/predict', json={}, timeout=5)
wrong_size_response = requests.post(
    BASE_URL + '/predict', json={'features': [0.1]}, timeout=5
)
print('Missing features:', missing_response.status_code, missing_response.json())
print('Wrong feature count:', wrong_size_response.status_code, wrong_size_response.json())
assert missing_response.status_code == 400
assert wrong_size_response.status_code == 400

Missing features: 400 {'error': "JSON body must include a 'features' key."}
Wrong feature count: 400 {'error': "'features' must be a list containing exactly two values."}


## 6. Stop the test server

Stopping the child process keeps the notebook repeatable and prevents a stale server from occupying port 5050. A user can later run `python app.py` to keep the API available.

In [6]:
server_process.terminate()
server_process.wait(timeout=5)
print('Test server stopped cleanly.')

Test server stopped cleanly.


## Productization notes

- **Reusable artifact:** `model/model.pkl` separates training from inference.
- **Efficient serving:** `app.py` loads the model once at startup.
- **Two clients:** POST supports programs sending JSON; GET supports a simple URL call.
- **Failure behavior:** invalid inputs produce explicit JSON errors with HTTP 400.
- **Scope:** this is an educational local API; production use would also require authentication, schema/version management, monitoring, rate limits, and a production WSGI server.